# Vicilin Molecular Dynamics Pipeline

**Protein:** pea vicilin homodimer (UniProt accession: _add here, same as in `docs/methods.md`_): two chains of 415 residues; the input PDB has 6,680 atoms (3,340 per chain) and 13,283 atoms after `pdb2gmx` adds hydrogens
**Pipeline:** AlphaFold3 structure → GROMACS MD → trajectory analysis → PyMOL visualization
**Conditions:** 300 K and 400 K, each analysed over a 200 ps and a 500 ps window (the 200 ps window is the first 200 ps of the 500 ps trajectory)
**Environment:** Google Colab GPU; GROMACS 2026.3 (conda-forge CUDA build, thread-MPI, binary `gmx`); AMBER99SB-ILDN force field; SPC/E water; **neutralizing Na⁺ only (no added salt)**
**System (from the original run):** cubic box 12.53 nm, 193,030 atoms = 13,283 protein + 179,730 water atoms (59,910 molecules) + 17 Na⁺

---

### Read this first: provenance and known caveats

1. **Assembled, not re-executed.** This notebook was assembled from several working sessions and cleaned. It has not been run end-to-end in exactly this form. Places where it differs from what was originally typed are marked **[changed]** or **[new]**. Unlike the legumin notebook, the NVT, NPT and production commands below are the real ones from the original session (only file names were standardised).
2. **GROMACS version.** Every simulation step (minimization, NVT, NPT, production, extension) ran on GROMACS 2026.3 (conda-forge, CUDA). Later post-processing sessions used a non-CUDA 2026.3 build, which does not affect analysis results. No cross-version restart occurred for vicilin.
3. **Equilibration used random initial velocities** (`gen_vel = yes`, `gen_seed = -1`), so a re-run will not be bit-identical.
4. **Vicilin and legumin setups are not identical.** Vicilin: no added salt (17 Na⁺ only, about 14 mM counter-ions), 12.53 nm box, C-rescale barostat, `DispCorr = EnerPres`, 2 ps output interval, minimization converged to Fmax < 1000. Legumin: 0.15 M NaCl (1,238 ions), 18.94 nm box, Parrinello-Rahman barostat, no explicit dispersion correction, 1 ps output interval, minimization stopped at machine precision. Cross-protein comparisons are therefore qualitative; the different ionic strength alone can change electrostatic screening and compaction.
5. **Single trajectory per condition, 200-500 ps.** Results are hypothesis-generating; no replicates.
6. **Not in the original notebook history, added here [new]:** the DSSP step (the DSSP files in `simulations/vicilin/` were produced in a separate session) and the 400 K movie (the repository's 400 K GIF came from another session). Protein-only fitted trajectories (`final_<T>.*`) are also new.
7. **Not in this notebook:** AlphaFold3 structure prediction (AlphaFold Server) and the confidence/chain-coloured prediction images (rendered in PyMOL on a local machine, see `alphafold_predictions/`).

**Runtime reference:** the 300 ps extension took about 11.6 min wall time at 300 K (about 37 ns/day) and 11.9 min at 400 K (about 36 ns/day). The GPU model was not recorded in the original notebook (CUDA driver 13.0, runtime 12.9).

### File-name map (original session → this notebook)

| Original session | This notebook |
|---|---|
| `nvt.*`, `npt.*` (300 K) | `nvt_300K.*`, `npt_300K.*` |
| `md.*` (300 K, 0-200 ps) | `md_300K_200ps.*` |
| `md_500ps.tpr`, `md_500ps.part0002.*`, `md_500ps_full.xtc` | `md_300K_500ps.tpr`, `md_300K_ext.part0002.*`, `md_300K_500ps.xtc` |
| `md_400K.*` | `md_400K_200ps.*` |
| `md_400K_500ps.part0002.*`, `md_400K_500ps_full.xtc` | `md_400K_ext.part0002.*`, `md_400K_500ps.xtc` |
| `gyrate_*.xvg`, `hbnum_*.xvg` | `rg_*.xvg`, `hbonds_*.xvg` |

## 1. Environment setup

`condacolab.install()` **restarts the runtime** automatically. After it restarts, continue from the next cell (do not re-run this one).

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CUDA build of GROMACS 2026.3 (conda-forge). In the original vicilin session this resolved to the
# thread-MPI CUDA build (binary: gmx). Pin version AND build: plain `gromacs=2026.3` can give an
# OpenCL build, and `gromacs=*=*cuda*` without a version resolved to 2024.5 in another session.
!mamba install -y -c conda-forge -c bioconda "gromacs=2026.3=*cuda*"
# If this does not give a CUDA build, this OpenMPI build worked in the original legumin session (binary: gmx_mpi):
# !mamba install -y -c conda-forge -c bioconda "gromacs=2026.3=mpi_openmpi_cuda_h56f47eb_0"

In [ ]:
# If only `gmx_mpi` exists (OpenMPI build), a symlink lets every command below use `gmx`.
!which gmx || ln -sf $(which gmx_mpi) /usr/local/bin/gmx
!gmx --version | grep -E "GROMACS version|GPU support|MPI library|CUDA (driver|runtime)"
!nvidia-smi --query-gpu=name,driver_version --format=csv
# Expected: GROMACS version 2026.3-conda_forge, MPI library thread_mpi, GPU support CUDA
!gmx --version | grep -q "GPU support: *CUDA" || echo "WARNING: this is not a CUDA build of GROMACS"

In [ ]:
import importlib
for m in ["numpy", "matplotlib", "pandas"]:
    try:
        importlib.import_module(m)
    except ImportError:
        print("missing:", m, "-> run: !pip install", m)
print("python libraries checked")

### Input files

The AlphaFold3 dimer (`structures/vicilin_dimer.pdb`) and all `.mdp` files come from the repository (`simulation_setup/vicilin/`). Always run `mdrun` from the local Colab disk (`/content/run`), not from the Drive mount. Drive is only used for backups (optional).

In [ ]:
import os
os.makedirs("/content/run", exist_ok=True)
!git clone --depth 1 https://github.com/malikahmedhassanawan/Legumin-Vicilin-Comparative-MD /content/repo
!cp /content/repo/structures/vicilin_dimer.pdb /content/run/
!cp /content/repo/simulation_setup/vicilin/*.mdp /content/run/
os.chdir("/content/run")
!ls

In [ ]:
# Optional: back up results to Drive after each stage (Colab sessions can end without warning).
from google.colab import drive
drive.mount("/content/drive")
import glob, shutil
BACKUP = "/content/drive/MyDrive/Vicilin_MD_backup"

def backup(patterns=("*.xtc", "*.cpt", "*.tpr", "*.log", "*.gro", "*.edr", "*.xvg", "*.dat", "*.top", "*.itp", "*.ndx")):
    os.makedirs(BACKUP, exist_ok=True)
    for p in patterns:
        for f in glob.glob(p):
            shutil.copy2(f, BACKUP)
    print("backed up to", BACKUP)

## 2. System preparation

Starting structure: the AlphaFold3-predicted homodimer (`vicilin_dimer.pdb` = `structures/vicilin_dimer.pdb`).

In [ ]:
# Topology: AMBER99SB-ILDN force field, SPC/E water. (Unlike legumin, no -ignh was used.)
!gmx pdb2gmx -f vicilin_dimer.pdb -o vicilin_processed.gro -water spce -ff amber99sb-ildn
# Expected: 2 chains (415 residues each) = 830 residues; 6,680 input atoms -> 13,283 atoms with hydrogens.
# Writes topol.top plus topol_Protein_chain_A/B.itp and posre_Protein_chain_A/B.itp (all needed later).

In [ ]:
# Cubic box with 1.0 nm padding
!gmx editconf -f vicilin_processed.gro -o vicilin_box.gro -c -d 1.0 -bt cubic
# Expected: solute diameter 10.53 nm -> cubic box 12.53 nm (volume 1,967 nm^3).

In [ ]:
!gmx solvate -cp vicilin_box.gro -cs spc216.gro -o vicilin_solv.gro -p topol.top
# Expected: 59,927 water molecules; 193,064 atoms.

In [ ]:
# Neutralize. The system charge is -17 e, so 17 Na+ are added and no Cl-.
!gmx grompp -f ions.mdp -c vicilin_solv.gro -p topol.top -o ions.tpr
# [changed] The solvent group is chosen by name (it was group 13 "SOL" in the original run).
# NOTE: no `-conc` option was used, i.e. NO added salt (about 14 mM Na+ counter-ions only).
# Legumin used `-conc 0.15` (0.15 M NaCl). Adding `-conc 0.15` here would not reproduce the repository data.
!echo "SOL" | gmx genion -s ions.tpr -o vicilin_ions.gro -p topol.top -pname NA -nname CL -neutral
# Expected: 17 NA + 0 CL replace 17 waters -> 59,910 waters, 193,030 atoms in total.

In [ ]:
# Sanity check: [ molecules ] must list both protein chains, SOL and NA.
!tail -6 topol.top

## 3. Energy minimization

`em.mdp`: steepest descent, `emtol` 1000 kJ/mol/nm, up to 50,000 steps, PME, 1.0 nm cutoffs.

**Original result:** converged to Fmax < 1000 in 1,591 steps (Epot = -3.3577 x 10^6 kJ/mol; Fmax = 985.8 kJ/mol/nm on atom 7213).

In [ ]:
!gmx grompp -f em.mdp -c vicilin_ions.gro -p topol.top -o em.tpr
!gmx mdrun -deffnm em -nb gpu
!grep -E "Potential Energy|Maximum force|converged" em.log
backup()

## 4. NVT and NPT equilibration

Both stages are 50,000 steps = **100 ps** (time step 2 fs), run for 300 K and 400 K from the same minimized structure.

| Setting | NVT | NPT |
|---|---|---|
| Position restraints | `-DPOSRES` | `-DPOSRES` |
| Thermostat | V-rescale, tau_t 0.1 ps, Protein / Non-Protein, `ref_t` = T | same |
| Barostat | none | C-rescale, tau_p 2.0 ps, 1 bar, isotropic, 4.5 x 10^-5 bar^-1 |
| Initial velocities | generated at T (`gen_vel = yes`, seed -1) | read from the NVT checkpoint |
| Other | LINCS on h-bonds, PME (order 4, spacing 0.16 nm), 1.0 nm cutoffs, `DispCorr = EnerPres` | same |

The `.mdp` files in `simulation_setup/vicilin/` are authoritative.

In [ ]:
# 300 K
!gmx grompp -f nvt_300K.mdp -c em.gro -r em.gro -p topol.top -o nvt_300K.tpr
!gmx mdrun -deffnm nvt_300K -nb gpu -pme gpu -bonded gpu
!gmx grompp -f npt_300K.mdp -c nvt_300K.gro -r nvt_300K.gro -t nvt_300K.cpt -p topol.top -o npt_300K.tpr -maxwarn 1
!gmx mdrun -deffnm npt_300K -nb gpu -pme gpu -bonded gpu
backup()

In [ ]:
# 400 K (starts again from em.gro)
!gmx grompp -f nvt_400K.mdp -c em.gro -r em.gro -p topol.top -o nvt_400K.tpr
!gmx mdrun -deffnm nvt_400K -nb gpu -pme gpu -bonded gpu
!gmx grompp -f npt_400K.mdp -c nvt_400K.gro -r nvt_400K.gro -t nvt_400K.cpt -p topol.top -o npt_400K.tpr -maxwarn 1
!gmx mdrun -deffnm npt_400K -nb gpu -pme gpu -bonded gpu
backup()

## 5. Production MD

Time step 2 fs; 200 ps = 100,000 steps; 500 ps = 250,000 steps. Compressed coordinates are written every 1,000 steps (**2 ps**): 101 frames after 200 ps, 251 after 500 ps. Position restraints are off, `continuation = yes`, `gen_vel = no`. The `.mdp` files differ between the two temperatures only in `ref_t`.

Each temperature was run to 200 ps and then extended by 300 ps with `gmx convert-tpr -extend 300`. The extension used `-noappend`, which writes a new set of `*.part0002.*` files that are joined with `gmx trjcat` (frames 0 to 500 ps).

In [ ]:
# 300 K: 200 ps production
!gmx grompp -f md_300K_200ps.mdp -c npt_300K.gro -t npt_300K.cpt -p topol.top -o md_300K_200ps.tpr
!gmx mdrun -deffnm md_300K_200ps -nb gpu -pme gpu -bonded gpu
backup()

In [ ]:
# 300 K: extend to 500 ps (about 11.6 min on the original GPU), keep part files, join with the first 200 ps
!gmx convert-tpr -s md_300K_200ps.tpr -extend 300 -o md_300K_500ps.tpr
!gmx mdrun -deffnm md_300K_ext -s md_300K_500ps.tpr -cpi md_300K_200ps.cpt -nb gpu -pme gpu -bonded gpu -noappend
!gmx trjcat -f md_300K_200ps.xtc $(ls md_300K_ext.part*.xtc) -o md_300K_500ps.xtc
!gmx check -f md_300K_500ps.xtc 2>&1 | grep -E "Last frame|# Atoms"
# Expected: 193030 atoms, 251 frames (every 2 ps), last frame at 500 ps
backup()

In [ ]:
# 400 K: 200 ps production
!gmx grompp -f md_400K_200ps.mdp -c npt_400K.gro -t npt_400K.cpt -p topol.top -o md_400K_200ps.tpr
!gmx mdrun -deffnm md_400K_200ps -nb gpu -pme gpu -bonded gpu
backup()

In [ ]:
# 400 K: extend to 500 ps (about 11.9 min on the original GPU)
!gmx convert-tpr -s md_400K_200ps.tpr -extend 300 -o md_400K_500ps.tpr
!gmx mdrun -deffnm md_400K_ext -s md_400K_500ps.tpr -cpi md_400K_200ps.cpt -nb gpu -pme gpu -bonded gpu -noappend
!gmx trjcat -f md_400K_200ps.xtc $(ls md_400K_ext.part*.xtc) -o md_400K_500ps.xtc
!gmx check -f md_400K_500ps.xtc 2>&1 | grep -E "Last frame|# Atoms"
backup()

In [ ]:
# Every log should report GROMACS 2026.3 (a log with two version lines was restarted on another version)
!grep -H "GROMACS version" *.log

## 6. Trajectory processing

Two processed versions of each 500 ps trajectory are made:

- `md_<T>_500ps_center.xtc`: whole system, molecules made whole and centred on the protein, used for RMSD, Rg, SASA, H-bonds and RMSF (same command as the original sessions, without `-ur compact`).
- `final_<T>.xtc` / `final_<T>.pdb` **[new]**: protein only, centred and then fitted (rotation + translation), used for DSSP and the PyMOL movies. The PDB (frame 0) carries the chain IDs.

Group numbers in this system: 0 System, 1 Protein, 4 Backbone.

In [ ]:
import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print(r.stderr[-1500:])
    return r

TEMPS = ["300K", "400K"]
WINDOWS = {"200ps": 200, "500ps": 500}   # analysis windows (ps), both start at 0

for T in TEMPS:
    tpr, xtc = f"md_{T}_500ps.tpr", f"md_{T}_500ps.xtc"
    sh(f'echo "1 0" | gmx trjconv -s {tpr} -f {xtc} -o md_{T}_500ps_center.xtc -pbc mol -center')
    sh(f'printf "1\\n1\\n" | gmx trjconv -s {tpr} -f {xtc} -o step1_{T}.xtc -pbc mol -center')
    sh(f'printf "1\\n1\\n" | gmx trjconv -s {tpr} -f step1_{T}.xtc -o final_{T}.xtc -fit rot+trans')
    sh(f'echo "1" | gmx trjconv -s {tpr} -f final_{T}.xtc -o final_{T}.pdb -dump 0')
    print(T, "processed")
backup()

## 7. Analysis

Whole-dimer metrics: RMSD of the backbone (fit and output group 4), radius of gyration and SASA of the protein (group 1), **protein-protein hydrogen bonds** (`-r "Protein" -t "Protein"`: 2,508 acceptors and 1,251 donors, bonds within and between the two chains), and per-residue RMSF. Time is written in **ps** for every metric.

> **[changed] Time unit.** The first original RMSD run used `-tu ns`, later ones `-tu ps`; `gyrate` always writes ps. Everything is ps here. (Some files in the repository are in ns and some in ps; the statistics scripts below detect this automatically.)

The 200 ps window is the first 200 ps of the 500 ps trajectory (`-b 0 -e 200`). Frames are 2 ps apart for vicilin (1 ps for legumin).

> **RMSF group.** Whole-dimer RMSF is written twice: `rmsf_<c>_allatom.xvg` (group 1, all protein atoms; what the original vicilin runs used) and `rmsf_<c>_backbone.xvg` (group 4; what most original legumin files used). Compare RMSF between proteins only on the same group.

In [ ]:
for T in TEMPS:
    tpr, ctr = f"md_{T}_500ps.tpr", f"md_{T}_500ps_center.xtc"
    for W, end in WINDOWS.items():
        c, opt = f"{T}_{W}", f"-b 0 -e {end}"
        sh(f'echo "4 4" | gmx rms -s {tpr} -f {ctr} -o rmsd_{c}.xvg -tu ps {opt}')
        sh(f'echo "1" | gmx gyrate -s {tpr} -f {ctr} -o rg_{c}.xvg {opt}')
        sh(f'echo "1" | gmx sasa -s {tpr} -f {ctr} -o sasa_{c}.xvg {opt}')
        sh(f'gmx hbond -s {tpr} -f {ctr} -num hbonds_{c}.xvg -r "Protein" -t "Protein" {opt}')
        sh(f'echo "1" | gmx rmsf -s {tpr} -f {ctr} -o rmsf_{c}_allatom.xvg -res {opt}')
        sh(f'echo "4" | gmx rmsf -s {tpr} -f {ctr} -o rmsf_{c}_backbone.xvg -res {opt}')
        print("done", c)
backup()

### Per-chain RMSF

`splitch 1` splits the Protein group into two groups named `Protein_chain1` (chain A) and `Protein_chain2` (chain B). The per-chain files use all protein atoms of each chain.

In [ ]:
for T in TEMPS:
    tpr, ctr = f"md_{T}_500ps.tpr", f"md_{T}_500ps_center.xtc"
    sh(f'printf "splitch 1\\nq\\n" | gmx make_ndx -f {tpr} -o chains_{T}.ndx')
    for W, end in WINDOWS.items():
        c = f"{T}_{W}"
        sh(f'echo "Protein_chain1" | gmx rmsf -s {tpr} -f {ctr} -n chains_{T}.ndx -o rmsf_chainA_{c}.xvg -res -b 0 -e {end}')
        sh(f'echo "Protein_chain2" | gmx rmsf -s {tpr} -f {ctr} -n chains_{T}.ndx -o rmsf_chainB_{c}.xvg -res -b 0 -e {end}')
    print(T, "per-chain RMSF done")
backup()

In [ ]:
import numpy as np

def read_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(("#", "@")):
                continue
            if line.strip():
                data.append([float(x) for x in line.split()])
    return np.array(data)

# Peak RMSF per chain (whole-dimer all-atom file: first half = chain A, second half = chain B)
for T in TEMPS:
    r = read_xvg(f"rmsf_{T}_500ps_allatom.xvg")
    half = len(r) // 2
    for name, part in [("A", r[:half]), ("B", r[half:])]:
        k = np.argmax(part[:, 1])
        print(f"{T} chain {name}: peak at position {k + 1} within the chain, RMSF = {part[k, 1]:.3f} nm")

### Secondary structure (DSSP) [new]

`gmx dssp` is built into GROMACS (no separate `mkdssp` needed). It runs on the protein-only fitted trajectory. The DSSP files in `simulations/vicilin/` came from a separate session; this cell regenerates them (`.dat`, 251 frames).

In [ ]:
for T in TEMPS:
    sh(f'gmx dssp -f final_{T}.xtc -s md_{T}_500ps.tpr -o vicilin_{T}_500ps_dssp.dat')
    print(T, "DSSP done")
backup()

## 8. Within-run statistics

Block statistics for Rg, SASA and H-bonds. The first 25% of each series is discarded as equilibration; 5-block averaging gives the SEM; Δ is the mean of the last 10% minus the mean of the first 10% (relative to the first-10% mean); `delta_over_SD` compares Δ to the run's own fluctuation.

> These are **within-run** errors from one trajectory. Consecutive frames are correlated and a drifting series inflates SD and SEM, so `delta_over_SD` is a descriptive measure, not a significance test. They do not replace independent replicates.

In [ ]:
import pandas as pd

EQUIL, NBLOCKS, EDGE = 0.25, 5, 0.10
out = []
for T in TEMPS:
    for W in WINDOWS:
        c = f"{T}_{W}"
        for m in ["rg", "sasa", "hbonds"]:
            y = read_xvg(f"{m}_{c}.xvg")[:, 1]
            n = max(int(len(y) * EDGE), 2)
            first, last = y[:n].mean(), y[-n:].mean()
            delta = last - first
            yp = y[int(len(y) * EQUIL):]
            bm = np.array([b.mean() for b in np.array_split(yp, NBLOCKS)])
            sem = bm.std(ddof=1) / np.sqrt(NBLOCKS)
            sd = yp.std(ddof=1)
            out.append(dict(protein="vicilin", condition=c, metric=m, mean=yp.mean(),
                            SEM_block=sem, SD=sd, delta=delta,
                            delta_pct=100 * delta / first, delta_over_SD=abs(delta) / sd))
df = pd.DataFrame(out).round(4)
df.to_csv("summary_table_stats_vicilin.csv", index=False)
print(df.to_string())

## 9. Quick visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for row, T in enumerate(TEMPS):
    rmsd = read_xvg(f"rmsd_{T}_500ps.xvg")
    rg = read_xvg(f"rg_{T}_500ps.xvg")
    axes[row, 0].plot(rmsd[:, 0], rmsd[:, 1])
    axes[row, 0].set_ylabel("Backbone RMSD (nm)")
    axes[row, 0].set_title(f"Vicilin RMSD, {T}, 500 ps")
    axes[row, 1].plot(rg[:, 0], rg[:, 1])
    axes[row, 1].set_ylabel("Rg (nm)")
    axes[row, 1].set_title(f"Vicilin radius of gyration, {T}, 500 ps")
    for ax in axes[row]:
        ax.axvline(200, ls=":", c="gray")   # boundary of the 200 ps window
for ax in axes[1]:
    ax.set_xlabel("Time (ps)")
plt.tight_layout()
plt.savefig("rmsd_rg_vicilin_300K_400K_500ps.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Trajectory visualization (PyMOL)

Cartoon with chain A in sky blue and chain B in salmon, exported frame by frame (no ray tracing, 1200 x 900) and assembled with `ffmpeg` into a GIF (800 px, 24 fps, as in the repository) and a smaller MP4 (1200 px). No hydrogen-bond overlay is drawn (the legumin movies have one).

> The 400 K movie **[new]** uses the same script as the original 300 K movie. Test with `MAX_FRAMES = 10` first.

In [ ]:
!pip install pymol-open-source --break-system-packages -q

In [ ]:
import pymol
from pymol import cmd
pymol.finish_launching(["pymol", "-qc"])

MAX_FRAMES = None   # e.g. 10 for a quick test; None = all frames (251)

for T in TEMPS:
    cmd.reinitialize()
    cmd.load(f"final_{T}.pdb", "traj")
    cmd.load_traj(f"final_{T}.xtc", "traj")
    print(T, "chains:", cmd.get_chains("traj"))   # expected ['A', 'B']; if empty, chain colours below will not apply
    cmd.bg_color("white")
    cmd.hide("everything")
    cmd.show("cartoon")
    cmd.color("skyblue", "chain A")
    cmd.color("salmon", "chain B")
    cmd.zoom("all", -28)
    os.makedirs(f"frames_{T}", exist_ok=True)
    n = cmd.count_states("traj")
    if MAX_FRAMES:
        n = min(n, MAX_FRAMES)
    for i in range(1, n + 1):
        cmd.frame(i)
        cmd.png(f"frames_{T}/frame_{i:04d}.png", width=1200, height=900, dpi=150, ray=0)
    print(T, "frames exported:", n)

In [ ]:
for T in TEMPS:
    sh(f'cd frames_{T} && ffmpeg -y -framerate 24 -i frame_%04d.png -vf "fps=24,scale=800:-1:flags=lanczos" ../vicilin_{T}_500ps.gif')
    sh(f'cd frames_{T} && ffmpeg -y -framerate 24 -i frame_%04d.png -vf "fps=24,scale=1200:-1:flags=lanczos" -pix_fmt yuv420p ../vicilin_{T}_500ps.mp4')
!ls -lh vicilin_*_500ps.gif vicilin_*_500ps.mp4
backup(("*.gif", "*.mp4"))

## 11. Record the software versions of this run

Run this at the end of a session and keep the output with your results.

In [ ]:
!gmx --version | head -12
!python --version
!conda list gromacs
!pip freeze | grep -iE "^(numpy|pandas|matplotlib|pymol)" | tee requirements_vicilin.txt

## Limitations

- One trajectory per temperature, 200-500 ps, no replicates; within-run error bars only.
- Homodimer only (not the native trimer/hexamer assembly); no experimental anchor.
- No added salt (17 Na⁺ only); legumin used 0.15 M NaCl, so ionic strength differs between the two proteins.
- Legumin and vicilin used different pressure coupling, dispersion-correction, box size and output-interval settings, so cross-protein comparisons are qualitative.
- Equilibration used random initial velocities; runs are not bit-reproducible.
- DSSP and the 400 K movie were not part of the original notebook history.